
<p align="center"> In the Name of God </p>

<table style="width: 100%; text-align: center;">
  <tr>
    <td style="width: 20%;"><img src="https://i.ibb.co/yXKQmtZ/logo1.png" alt="Left Logo" width="100"></td>
    <td style="width: 60%;">
      <p>Faculty of Electrical and Computer Engineering</p>
      <p>Interactive learning - Fall 2023-2024</p>
      <p>Assignment Series 2</p>
      <p>Markov Decision Processes and Dynamic programming</p>
    </td>
    <td style="width: 20%;"><img src="https://i.ibb.co/wLjqFkw/logo2.png" alt="Right Logo" width="100"></td>
  </tr>
  
</table>
<p align="center">Fatemeh Mohammadi - 810199489</p>
<hr>



In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve
from sympy import symbols, solveset, Interval, nsolve
import time
import timeit
import pandas as pd


# <font color = lightGreen> Problem 3: 
 </font>

In [ ]:
import numpy as np

# Real States
states_in_real = list(range(10))

# Transition Probabilities P _ in real: # Base on Real States
def transitions_in_real(current_real_state, a): # P(current_s,a) -> {next_s: probability}
    next_s = current_real_state
    probability = 1  # Deterministic in the real world
    if current_real_state in [7, 8, 9]:
        return {next_s: probability}
    if a == 0:  # Right
        if current_real_state in [0, 1, 2, 3, 4, 5]:
            next_s = current_real_state + 1
        else:
            next_s = current_real_state

    elif a == 1:  # Up
        if current_real_state == 1:
            next_s = 8
        elif current_real_state == 3:
            next_s = 7
        elif current_real_state == 5:
            next_s = 9
        else:
            next_s = current_real_state

    elif a == 2:  # Left
        if current_real_state in [1, 2, 3, 4, 5, 6]:
            next_s = current_real_state - 1

    else:  # NO VALID ACTION -> will not move
        next_s = current_real_state

    return {next_s: probability}

# Observations
def get_observation(current_real_state):
    if current_real_state == 0:  # O1
        return [1, 0, 0, 0], 0
    elif current_real_state in [1, 3, 5]:  # O2, O4, O6
        return [1, 1, 1, 0], 1
    elif current_real_state in [2, 4]:  # O3, O5
        return [1, 0, 1, 0], 2
    elif current_real_state == 6:  # O7
        return [0, 0, 1, 0], 3
    else:  # Terminals
        return [0, 0, 0, 1], 4

# Reward Function
def rewards_in_real(next_s):
    if next_s == 8 or next_s == 9:
        return 1
    elif next_s == 7:
        return -1
    return 0

# Environment Step
def environment_step(current_real_state, action):
    next_real_state, prob = list(transitions_in_real(current_real_state, action).items())[0]
    observation, abstract_next_state = get_observation(next_real_state)
    reward = rewards_in_real(next_real_state)
    return next_real_state, abstract_next_state, reward

# Value Iteration
def value_iteration_with_environment(environment_step, abstract_states, actions, gamma, theta):
    """
    Perform value iteration using the environment step function.
    """
    num_states = len(abstract_states)
    V = np.zeros(num_states)  # Initialize value function
    policy = np.zeros(num_states, dtype=int)  # Initialize policy

    while True:
        delta = 0  # Convergence criteria
        for s in abstract_states:
            v = V[s]
            # Compute value for each action and select the best
            action_values = np.zeros(len(actions))
            for a in actions:
                total_value = 0
                for real_state in states_in_real:
                    _, abstract_next_state, reward = environment_step(real_state, a)
                    total_value += reward + gamma * V[abstract_next_state]
                action_values[a] = total_value / len(states_in_real)
            V[s] = max(action_values)
            policy[s] = np.argmax(action_values)
            delta = max(delta, abs(v - V[s]))

        # if delta < theta:
            # break
    return V, policy

# Parameters
gamma = 0.9  # Discount factor
theta = 1e-6  # Convergence threshold
P = 2 / 3  # Transition probability

# Initialize abstract states and actions
abstract_states = list(range(5))  # Abstract states (O1 to O7 mapped to 5 abstract states)
actions = [0, 1, 2]  # Right (0), Up (1), Left (2)

# Perform Value Iteration
optimal_value_function, optimal_policy = value_iteration_with_environment(
    environment_step, abstract_states, actions, gamma, theta
)

# Display results
import pandas as pd
df = pd.DataFrame({
    "Abstract State": abstract_states,
    "Value Function": optimal_value_function,
    "Policy": optimal_policy
})

print(df)


   Abstract State  Value Function  Policy
0               0        1.999994       1
1               1        1.999994       1
2               2        1.999994       1
3               3        1.999994       1
4               4        1.999994       1


## Define Environment:

In [14]:
## Real States:
    #0,  1,  2,  3,  4,  5,  6,  7,     8,           ,9
    #O1, O2, O3, O4, O5, O6, O7, RED_T, Green_T(Left), Green_T(Right)
states_in_real = list(range(10))

## Transition Probabilities P _ in real: # Base on Real States
def transitions_in_real(current_real_state, a): #P(current_s,a) -> {next_s: probability}
    next_s = current_real_state
    probability = 1 # Deterministic -> in real world
    if a == 0: #Right
        if current_real_state in [0,1,2,3,4,5]:
            next_s = current_real_state + 1
        else:
            next_s = current_real_state

    elif a == 1: #Up
        if current_real_state == 1:
            next_s = 8
        elif current_real_state == 3:
            next_s = 7
        elif current_real_state == 5:
            next_s = 9
        else:
            next_s = current_real_state

    elif a == 2: #Left
        if current_real_state in [1,2,3,4,5,6]:
            next_s = current_real_state - 1

    else: # NO VALID ACTION -> will not move
        next_s = current_real_state

    return {next_s: probability}

## Observations:
#0 -> Wall/ 1->Free
sensors_output = [None, None, None, None] #[R,U,L,D]
def get_observation(current_real_state):
    sensors_output = [None, None, None, None]
    if current_real_state == 0: #O1
        sensors_output = [1, 0, 0, 0]
    elif current_real_state in [1, 3, 5]: #O2, O4, O6
        sensors_output = [1, 1, 1, 0]
    elif current_real_state in [2, 4]: #O3, O4
        sensors_output = [1, 0, 1, 0]
    elif current_real_state == 6: #O7
        sensors_output = [0, 0, 1, 0]
    else: #Terminals
        sensors_output = [0, 0, 0, 1]
    return sensors_output

## Reward Function R _ in real: # Base on Real States
def rewards_in_real(next_s):
    reward = 0
    if next_s == 8 or next_s == 9:
        reward = 1

    elif next_s == 7:
        reward = -1

    return reward



In [ ]:
def environment_step(current_real_state, action):
    next_real_state, prob = (list(transitions_in_real(current_real_state, action).items()))[0]
    observation = get_observation(next_real_state)
    reward = rewards_in_real(next_real_state)
    return next_real_state, observation, reward

## Agent:

In [ ]:
## State Space S:
#0: O1
#1: O2,O4,O6
#2: O3,O5
#3: O7
#4: Terminals
abstract_states = list(range(5))

## Action Space A:
#0->Right, 1->UP, 2->Left
actions = list(range(3))

## Abstract P:
P = 2/3
transition_matrix = {
    0: np.array([  # Right action
        [0, 1, 0, 0, 0],
        [0, 0, 1-P, P, 0],
        [0, 1, 0, 0, 0],
        [0, 0, 0, 1, 0],
        [0, 0, 0, 0, 1]
    ]),
    1: np.array([  # Up action
        [1, 0, 0, 0, 0],
        [0, 0, 0, 0, 1],
        [0, 0, 1, 0, 0],
        [0, 0, 0, 1, 0],
        [0, 0, 0, 0, 1]
    ]),
    2: np.array([  # Left action
        [1, 0, 0, 0, 0],
        [1-P,0, P, 0, 0],
        [0, 1, 0, 0, 0],
        [0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1]
    ])
}
def abstract_transitions(belief, action):
    next_belief = np.dot(belief, transition_matrix[action])
    return next_belief


In [ ]:
gamma = 0.9  # (or Lambda) -> Discount factor
theta = 1e-6  # threshod for Convergence

In [ ]:
def update_belief(belief, action, observation):
    next_belief = abstract_transitions(belief, action)

    #Likelihood function: map observations -> abstract states
    likelihood = np.zeros(len(abstract_states))
    if observation == [1, 0, 0, 0]:
        likelihood = [1, 0, 0, 0, 0]
    elif observation == [1, 1, 1, 0]:
        likelihood = [0, 1, 0, 0, 0]
    elif observation == [1, 0, 1, 0]:
        likelihood = [0, 0, 1, 0, 0]
    elif observation == [0, 0, 1, 0]:
        likelihood = [0, 0, 0, 1, 0]
    elif observation == [0, 0, 0, 1]:
        likelihood = [0, 0, 0, 0, 1]

    next_belief = next_belief * likelihood
    if next_belief.sum() > 0:
        next_belief /= next_belief.sum()  # Normalize
    return next_belief


In [ ]:
def value_iteration(states, actions, gamma=0.9, theta=1e-6):
    V = np.zeros(len(states))  # Initialize value function
    policy = np.zeros(len(states), dtype=int)  # Initialize policy

    while True:
        delta = 0  # Track maximum change
        for s in states:
            action_values = []  # Store values for all actions in state s
            for a in actions:
                # Determine the deterministic next state
                # Get data from enviroment TODO

                # Compute value for this action
                value = gamma * V[next_abstract_state]
                action_values.append(value)

            # Update value and policy for state s
            best_action_value = max(action_values)
            delta = max(delta, abs(best_action_value - V[s]))
            V[s] = best_action_value
            policy[s] = np.argmax(action_values)

        # Check for convergence
        if delta < theta:
            break

    return policy


In [ ]:
def simulate_policy(policy, start_real_state, steps=10):
    current_real_state = start_real_state
    belief = np.array([0.2 for i in abstract_states])  # Start belief in state 0
    total_reward = 0
    history = []

    for _ in range(steps):
        # Choose action based on belief (using most likely abstract state)
        most_likely_state = np.argmax(belief)
        print(most_likely_state, belief)
        action = policy[most_likely_state]

        # Step in environment
        next_real_state, observation, reward = environment_step(current_real_state, action)
        total_reward += reward

        # Update belief based on observation
        belief = update_belief(belief, action, observation)

        # Record step data
        history.append((current_real_state, action, next_real_state, observation, reward))
        current_real_state = next_real_state

    return history, total_reward


In [ ]:
optimal_policy = value_iteration(abstract_states, actions)
start_real_state = np.random.randint(10)
num_steps = 20
simulation_history, simulation_total_reward = simulate_policy(optimal_policy, start_real_state, num_steps)

# Result:
result_simulation = pd.DataFrame(simulation_history, columns=["Current Real State", "Action", "Next Real State", "Observation", "Reward"])
result_simulation["Total Reward"] = simulation_total_reward
result_simulation

# <font color = lightGreen> Problem 4: </font>

## PART 1: (Questions)

### 4.1.D

In [ ]:
def equation_1(lambda_):
    return 1 + lambda_ + lambda_**2 - (100 * lambda_**6)

lambda_values = np.linspace(0, 1, 500)
equation_values = equation_1(lambda_values)

plt.plot(lambda_values, equation_values, label=r"$1 + \lambda + \lambda^2 - 100\lambda^6$")
plt.axhline(0, color='red', linestyle='--', label="y = 0")
plt.xlabel(r"$\lambda$", fontsize=14)
plt.ylabel("Equation Value", fontsize=14)
plt.title("Plot of the Equation", fontsize=16)
plt.legend(fontsize=12)
plt.grid(True)
plt.show()


In [ ]:
lambda_sym = symbols('lambda', positive=True, real=True)
equation_1 = 1 + lambda_sym + lambda_sym**2 - 100 * lambda_sym**6

root1 = nsolve(equation_1, lambda_sym, 0.9)
root1

In [ ]:
def equation_2(lambda_):
    return 1 + lambda_ + lambda_**3 - (100 * lambda_**6) + (100 * lambda_**8)

lambda_values = np.linspace(0, 1, 500)
modified_equation_values = equation_2(lambda_values)

plt.plot(lambda_values, modified_equation_values, label=r"$1 + \lambda + \lambda^3 - 100\lambda^6 + 100\lambda^8$")
plt.axhline(0, color='red', linestyle='--', label="y = 0")
plt.xlabel(r"$\lambda$", fontsize=14)
plt.ylabel("Equation Value", fontsize=14)
plt.title("Plot of the Modified Equation", fontsize=16)
plt.legend(fontsize=12)
plt.grid(True)
plt.show()


In [ ]:
lambda_sym = symbols('lambda', positive=True, real=True)
equation_2 = 1 + lambda_sym + lambda_sym**3 - 100 * lambda_sym**6 + 100 * lambda_sym**8

roots = solveset(equation_2, lambda_sym, domain=Interval(0, 1))

root1 = nsolve(equation_2, lambda_sym, 0.5)
root2 = nsolve(equation_2, lambda_sym, 0.9)
(root1, root2)

## PART 2: (Simulation)

### Define MDP parameters:

In [ ]:
## State Space S:
states = list(range(11))

## Action Space A:
actions = ["Buy", "Sell"]

## Transition Probabilities P:
def transitions(current_s, a): #P(current_s,a) -> {next_s: probability = 1}
    next_s = 0
    probability = 1 # Deterministic

    if a == "Buy":
        if current_s < 10:
            next_s = current_s + 1
        else:
            next_s = current_s
    elif a == "Sell":
        if current_s > 0:
            next_s = current_s - 1
        else:
            next_s = current_s

    return {next_s: probability}


## Reward Function R:
def rewards(current_s, a, next_s):
    reward = 0
    if a == "Buy":
        if current_s == 9 and next_s == 10:
            reward = 100
        else:
            reward = 0
    elif a == "Sell":
        if current_s > 0 and next_s == current_s - 1:
            reward = 1
        else:
            reward = 0
    return reward


### 4.2.A.

#### Policy Iteration:

In [ ]:
def finite_horizon_policy_iteration(states, actions, H, transitions, rewards, gamma = 1):
    #0. Initialize random policy
    policy = { (s, t): (np.random.choice(actions) if s != 10 else None) for s in states for t in range(H) }

    while True:
        #1. Initialize value function
        V = { (s, t): 0 for s in states for t in range(H+1) }

        #2. Policy Evaluation
        for t in range(H - 1, -1, -1):  # Iterate backward from H->0
            for s in states:
                if s == 10:  # Terminal state
                    continue
                a = policy[(s, t)]
                value = 0
                for next_s, prob in transitions(s, a).items():
                    value += prob * (rewards(s, a, next_s) + gamma * V[(next_s, t+1)])
                V[(s, t)] = value


        #3. Policy Improvement
        stable = True
        for t in range(H):
            for s in states:
                if s == 10:  # Terminal state
                    continue

                best_action = None
                best_value = float('-inf')

                for a in actions:
                    value = 0
                    for next_s, prob in transitions(s, a).items():
                        value += prob * (rewards(s, a, next_s) + gamma * V[(next_s, t+1)])
                    if value > best_value:
                        best_value = value
                        best_action = a

                if policy[(s, t)] != best_action:
                    stable = False
                    policy[(s, t)] = best_action

        if stable:
            break  # Policy has converged

    return policy , V

In [ ]:
# For Debug:
H = 7
policy, V = finite_horizon_policy_iteration(states, actions, H, transitions, rewards)
V

#### Value Iteration:

In [ ]:
def finite_horizon_value_iteration(states, actions, H, transitions, rewards, gamma = 1):
    # 1. Initialize value function for all states and time steps
    V = { (s, t): 0 for s in states for t in range(H+1) }
    policy = { (s, t): None for s in states for t in range(H) }

    # 2. Perform value iteration backward from horizon H
    for t in range(H - 1, -1, -1):  # Iterate backward from H-1 to 0
        for s in states:
            if s == 10:  # Terminal state
                continue

            best_action = None
            best_value = float('-inf')

            #2.1. Evaluate each action
            for a in actions:
                value = 0
                for next_s, prob in transitions(s, a).items():
                    value += prob * (rewards(s, a, next_s) + gamma * V[(next_s, t+1)])

                if value > best_value:
                    best_value = value
                    best_action = a

            #2.2. Update the value function and policy
            V[(s, t)] = best_value
            policy[(s, t)] = best_action

    return policy, V


In [ ]:
# For Debug:
H = 7
policy, V = finite_horizon_value_iteration(states, actions, H, transitions, rewards)
V


#### Simulation

In [ ]:
def simulate_actions_for_horizon(H, start_state, states, actions, transitions, rewards, algorithm, gamma = 1):
    # Compute the optimal policy for the horizon

    policy, _ = algorithm(states, actions, H, transitions, rewards, gamma)

    # Simulate the trajectory
    current_state = start_state
    trajectory = []
    for t in range(H):
        if current_state == 10:  # Terminal state
            trajectory.append((t, current_state, None))
            break

        # Get the action from policy
        action = policy.get((current_state, t), None)
        trajectory.append((t, current_state, action))

        # Updating state
        next_states = transitions(current_state, action)
        current_state = list(next_states.keys())[0]  # Deterministic transition

    return trajectory


def simulate_and_categorize(Hs, start_state, states, actions, transitions, rewards, algorithm, gamma = 1):
    for H in Hs:
        # Simulate the trajectory for the given H
        trajectory = simulate_actions_for_horizon(H, start_state, states, actions, transitions, rewards, algorithm, gamma)

        buy_count = sum(1 for _, _, a in trajectory if a == "Buy")
        sell_count = sum(1 for _, _, a in trajectory if a == "Sell")

        # Categorize the policy
        if sell_count == 0:
            policy_type = "Buy-only policy"
        elif buy_count == 0:
            policy_type = "Sell-only policy"
        else:
            policy_type = "Mixed policy (buy and sell)"

        # Results
        print(f"H = {H},\t#Sell = {sell_count},\t#Buy = {buy_count},\t-> {policy_type}")
        for t, s, a in trajectory:
            print(f"Time Step: {t}, State: {s}, Action: {a}")
        print(10 * "___")


In [ ]:
Hs = [3, 5, 7, 8, 9, 12]
start_state = 3


In [ ]:
print("PI:")
simulate_and_categorize(Hs, start_state, states, actions, transitions, rewards, finite_horizon_value_iteration)


In [ ]:
print("VI:")
simulate_and_categorize(Hs, start_state, states, actions, transitions, rewards, finite_horizon_value_iteration)



#### Compare PI & VI:

In [ ]:
Hs = [np.random.randint(20)]
start = np.random.randint(10)
print("PI:")
simulate_and_categorize(Hs, start_state, states, actions, transitions, rewards, finite_horizon_value_iteration)
print("VI:")
simulate_and_categorize(Hs, start_state, states, actions, transitions, rewards, finite_horizon_value_iteration)


In [ ]:
execution_times_PI = []
execution_times_VI = []

for i in range(10):
    H = np.random.randint(100, 1000)
    execution_time_PI = timeit.timeit(lambda: finite_horizon_policy_iteration(states, actions, H, transitions, rewards), number=1)
    execution_time_VI = timeit.timeit(lambda: finite_horizon_value_iteration(states, actions, H, transitions, rewards), number=1)
    execution_times_PI.append(execution_time_PI)
    execution_times_VI.append(execution_time_VI)

avg_execution_time_PI = sum(execution_times_PI) / len(execution_times_PI)
avg_execution_time_VI = sum(execution_times_VI) / len(execution_times_VI)

print(f"Avg Execution Time - PI: {avg_execution_time_PI:.5f} seconds")
print(f"Avg Execution Time - VI: {avg_execution_time_VI:.5f} seconds")


### 4.2.B. 

In [ ]:
gamma = 0


#### Policy Iteration

In [ ]:
def infinite_horizon_policy_iteration(states, actions, transitions, rewards, gamma=0, epsilon=0.00001):
    # 0. Initialize random policy
    policy = {s: (np.random.choice(actions) if s != 10 else None) for s in states}

    while True:
        #1. Initialize value function
        V = {s: 0 for s in states}

        #2. Policy Evaluation
        while True:
            delta = 0
            for s in states:
                if s == 10:  # Terminal state
                    continue
                a = policy[s]
                value = 0
                for next_s, prob in transitions(s, a).items():
                    value += prob * (rewards(s, a, next_s) + gamma * V[next_s])
                delta = max(delta, abs(V[s] - value))
                V[s] = value
            if delta < epsilon:  # Value function has converged
                break

        #2. Policy Improvement
        stable = True
        for s in states:
            if s == 10:  # Terminal state
                continue

            best_action = None
            best_value = float('-inf')

            for a in actions:
                value = 0
                for next_s, prob in transitions(s, a).items():
                    value += prob * (rewards(s, a, next_s) + gamma * V[next_s])
                if value > best_value:
                    best_value = value
                    best_action = a

            if policy[s] != best_action:
                stable = False
                policy[s] = best_action

        if stable:
            break  # Policy has converged

    return policy, V


In [ ]:
policy, V= infinite_horizon_policy_iteration(states, actions, transitions, rewards, gamma=gamma, epsilon=0.00001)
policy

#### Value Iteration

In [ ]:
def infinite_horizon_value_iteration(states, actions, transitions, rewards, gamma=0, epsilon=0.00001):
    # 1. Initialize value function and policy for all states
    V = {s: 0 for s in states}
    policy = {s: None for s in states}

    while True:
        delta = 0
        # 2. Perform value iteration across all states
        for s in states:
            if s == 10:  # Terminal state
                continue

            best_action = None
            best_value = float('-inf')

            # 2.1 Evaluate each action
            for a in actions:
                value = 0
                for next_s, prob in transitions(s, a).items():
                    value += prob * (rewards(s, a, next_s) + gamma * V[next_s])

                if value > best_value:
                    best_value = value
                    best_action = a

            # 2.2 Update the value function and policy
            delta = max(delta, abs(V[s] - best_value))
            V[s] = best_value
            policy[s] = best_action

        # 3. Check for convergence
        if delta < epsilon:  # Value function has converged
            break

    return policy, V


In [ ]:
policy, V= infinite_horizon_value_iteration(states, actions, transitions, rewards, gamma=gamma, epsilon=0.00001)
policy

#### Compare PI & VI:

In [ ]:
print("PI:")
policy_PI, _ = infinite_horizon_policy_iteration(states, actions, transitions, rewards, gamma=gamma, epsilon=0.00001)
print(policy_PI)
print("VI:")
policy_VI, _ = infinite_horizon_value_iteration(states, actions, transitions, rewards, gamma=gamma, epsilon=0.00001)
print(policy_VI)


In [ ]:
execution_times_PI = []
execution_times_VI = []

for i in range(10):
    H = np.random.randint(100, 1000)
    execution_time_PI = timeit.timeit(lambda: infinite_horizon_policy_iteration(states, actions, transitions, rewards, gamma=gamma, epsilon=0.00001), number=1)
    execution_time_VI = timeit.timeit(lambda: infinite_horizon_value_iteration(states, actions, transitions, rewards, gamma=gamma, epsilon=0.00001), number=1)
    execution_times_PI.append(execution_time_PI)
    execution_times_VI.append(execution_time_VI)

avg_execution_time_PI = sum(execution_times_PI) / len(execution_times_PI)
avg_execution_time_VI = sum(execution_times_VI) / len(execution_times_VI)

print(f"Avg Execution Time - PI: {avg_execution_time_PI:.5f} seconds")
print(f"Avg Execution Time - VI: {avg_execution_time_VI:.5f} seconds")


### 4.2.C. 

In [ ]:
gammas = [0.0, 0.1, 0.5, 0.9]


#### Policy Iteration

In [ ]:
gammas = [0.0, 0.1, 0.5, 0.9]
for gamma in gammas:
    print(f"gamma = {gamma}:")
    policy, _ = infinite_horizon_policy_iteration(states, actions, transitions, rewards, gamma=gamma, epsilon=0.00001)
    print(policy)
    print("")


#### Value Iteration

In [ ]:
gammas = [0.0, 0.1, 0.5, 0.9]
for gamma in gammas:
    print(f"gamma = {gamma}:")
    policy, _ = infinite_horizon_value_iteration(states, actions, transitions, rewards, gamma=gamma, epsilon=0.00001)
    print(policy)
    print("")

#### Compare PI & VI:

In [ ]:
for i in range(5):
    gamma = np.random.uniform()
    print(f"gamma = {gamma}:")
    policy_VI, _ = infinite_horizon_value_iteration(states, actions, transitions, rewards, gamma=gamma, epsilon=0.00001)
    policy_PI, _ = infinite_horizon_policy_iteration(states, actions, transitions, rewards, gamma=gamma, epsilon=0.00001)
    if policy_PI == policy_VI:
        print("SAME")
        print("Policy:", policy_PI)
    else:
        print(f"policy_PI = {policy_PI}" )
        print(f"policy_VI = {policy_VI}" )

    print("")

In [ ]:
execution_times_PI = []
execution_times_VI = []

for i in range(10):
    gamma = np.random.uniform()
    H = np.random.randint(100, 1000)
    execution_time_PI = timeit.timeit(lambda: infinite_horizon_policy_iteration(states, actions, transitions, rewards, gamma=gamma, epsilon=0.00001), number=1)
    execution_time_VI = timeit.timeit(lambda: infinite_horizon_value_iteration(states, actions, transitions, rewards, gamma=gamma, epsilon=0.00001), number=1)
    execution_times_PI.append(execution_time_PI)
    execution_times_VI.append(execution_time_VI)

avg_execution_time_PI = sum(execution_times_PI) / len(execution_times_PI)
avg_execution_time_VI = sum(execution_times_VI) / len(execution_times_VI)

print(f"Avg Execution Time - PI: {avg_execution_time_PI:.5f} seconds")
print(f"Avg Execution Time - VI: {avg_execution_time_VI:.5f} seconds")
